In [ ]:
# FILE: 02_Run_Analysis.ipynb
# CELL 1: Setup and Environment

# --- Install Packages ---
print("Installing packages...")
%pip install pandas geopandas rasterio shapely fiona pyproj rapidfuzz xarray netCDF4 dask matplotlib seaborn -q

from google.colab import drive
import pandas as pd
import os

# --- Mount Drive ---
drive.mount('/content/drive', force_remount=True)
%cd "/content/drive/MyDrive/IndiaProject_Colab"

# --- Run Setup Script ---
print("\nLoading functions...")
%run 00_Functions_and_Setup.ipynb
print("Setup complete.")

Installing packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.6 MB/s eta 0:00:00
Mounted at /content/drive
/content/drive/MyDrive/IndiaProject_Colab

Loading functions...
Setup complete.


In [ ]:
# FILE: 02_Run_Analysis.ipynb
# CELL 2: Define Experiment Configurations

# --- Global Settings ---
YEAR_SHIFTS_TO_TEST = [-1, 0, 1]
# Note: MIN_YEARS_FOR_MODEL is defined in Notebook 00 (ensure it is set to 10 there)

# --- Define Features ---
# We use the final feature set including SIF
FEATURES_WITH_SIF = ["sm_season_mean", "tmax_season_mean", "csif_season_mean"]

# --- Generate Crop/Season Configs ---
EXPERIMENT_CONFIGS = []
print("--- Generating experiment configurations (MidPlant_to_MidHarvest) ---")

for crop, calendar in CROP_CALENDARS.items():
    p_start, p_end = calendar['planting_start'], calendar['planting_end']
    h_start, h_end = calendar['harvest_start'], calendar['harvest_end']

    # Calculate the season range
    months = get_month_range(get_middle_month(p_start, p_end), get_middle_month(h_start, h_end))

    EXPERIMENT_CONFIGS.append({
        "crop": crop,
        "season_name": f"{crop.replace('_', ' ').title()}_MidPlant_to_MidHarvest",
        "months": months
    })

print(f"Ready to run {len(EXPERIMENT_CONFIGS)} crop configurations.")

--- Generating experiment configurations (MidPlant_to_MidHarvest) ---
Ready to run 4 crop configurations.


In [ ]:
# FILE: 02_Run_Analysis.ipynb
# CELL 3: Final Analysis (4-Stage Hierarchical Model)

import pandas as pd
import numpy as np
import os

print("\n" + "="*80)
print(f"   STARTING PHASE 4: HIERARCHICAL ANALYSIS")
print(f"   Model: Trend -> Climate -> Irrigation -> SIF")
print(f"   Minimum Years Requirement: {MIN_YEARS_FOR_MODEL}")
print("="*80)

# --- 1. Load and Process Yield/Irrigation Data ---
print("Loading and processing crop data...")
if not os.path.exists(YLD_FILE):
    raise FileNotFoundError(f"Master Data File not found: {YLD_FILE}")

yld = pd.read_csv(YLD_FILE)

# --- Data Preparation Logic ---
# 1. Interpolate Irrigation gaps (Linear interpolation per district-crop)
print("  > Interpolating missing irrigation records...")
yld['irrig_area_1000ha'] = yld.groupby(['Dist Code', 'crop'])['irrig_area_1000ha'].transform(
    lambda group: group.interpolate(method='linear', limit_direction='both')
)

# 2. Calculate Irrigated Fraction
print("  > Calculating Irrigated Fraction (Area / Total Area)...")
# Replace 0 total area with NaN to avoid Inf
yld['total_area_temp'] = yld['total_area_1000ha'].replace(0, np.nan)
yld['irrigated_pct'] = yld['irrig_area_1000ha'] / yld['total_area_temp']

# 3. Clean Percentages
# Clip to [0, 1] range (handle reporting errors where irrig > total)
yld['irrigated_pct'] = yld['irrigated_pct'].clip(lower=0.0, upper=1.0)
# Fill NaNs with 0.0 (Assume rainfed if data missing but yield exists)
yld['irrigated_pct'] = yld['irrigated_pct'].fillna(0.0)

# Clean up temp column
yld.drop(columns=['total_area_temp'], inplace=True)
print("  > Data Prep Complete.")

# -----------------------------------------------
# RUN THE EXPERIMENT LOOP
# -----------------------------------------------

all_hierarchical_results = []

# We don't need to pass a feature list anymore;
# the new fit_district_model hardcodes the hierarchy groups.
# We just need to ensure the columns exist in the dataframe.

for year_shift in YEAR_SHIFTS_TO_TEST:
    print(f"\n--- Processing Shift: {year_shift} ---")

    for config in EXPERIMENT_CONFIGS:
        crop, months = config["crop"], config["months"]
        # print(f"   > Processing {crop}...")

        # 1. Get Climate Panels (Unweighted)
        # Returns: Dist Code, year, sm_season_mean, tmax_season_mean
        climate_panel = get_climate_panel_for_months_NO_WEIGHTING(months=months)

        # 2. Get SIF Panel (Weighted by Crop Area)
        # Returns: Dist Code, year, csif_season_mean
        sif_panel = get_sif_panel_for_months(crop_name=crop, months=months)

        # 3. Merge Environmental Data
        env_panel = pd.merge(climate_panel, sif_panel, on=["Dist Code", "year"], how="inner")

        # 4. Prepare Yield Data (Shifted)
        yld_shifted = yld[yld["crop"] == crop].copy()
        yld_shifted['year'] -= year_shift # Align Yield(t) with Env(t + shift)

        # 5. Merge Everything
        # Inner join on Env Data (Master) + Left Join Yield
        # (Actually, fit_district_model handles dropping NaNs, so Left Join is safer to keep env data visible)
        df_merged = env_panel.merge(yld_shifted, on=["Dist Code", "year"], how="left")

        # 6. Fit Hierarchical Models
        # The function now returns specific metrics: R2_1, R2_2, Mq_Climate, etc.
        results = []
        for dist_code, g in df_merged.groupby("Dist Code"):
            # fit_district_model is now the UPDATED version from Notebook 00
            metrics, _ = fit_district_model(g, features=None) # features arg is ignored in new version
            if metrics:
                results.append(metrics)

        if results:
            df_res = pd.DataFrame(results)
            df_res["year_shift"] = year_shift
            df_res["crop"] = crop
            all_hierarchical_results.append(df_res)
            # print(f"     [+] {len(df_res)} districts modeled.")

# --- Save Final Results ---
if all_hierarchical_results:
    final_df = pd.concat(all_hierarchical_results, ignore_index=True)

    # Define New Filename
    SAVE_FILE = os.path.join(RESULTS_DIR, "FINAL_HIERARCHICAL_INTERACTION_metrics.parquet")

    final_df.to_parquet(SAVE_FILE)
    print("\n" + "="*80)
    print(f" [✓] SUCCESS: Hierarchical results saved to:")
    print(f"     {SAVE_FILE}")
    print(f"     Total Records: {len(final_df)}")
    print("="*80)

    # Preview the columns to confirm the new metrics exist
    print("\nColumns Generated:")
    print(final_df.columns.tolist())

    # Quick Sanity Check on Marginal Gains
    if 'Mq_SIF' in final_df.columns:
        print("\nSanity Check (Mean Marginal Gains):")
        print(final_df[['Mq_Climate', 'Mq_Irrig', 'Mq_SIF']].mean())
else:
    print("\n[!] No results generated. Check input data.")


   STARTING PHASE 4: HIERARCHICAL ANALYSIS
   Model: Trend -> Climate -> Irrigation -> SIF
   Minimum Years Requirement: 15
Loading and processing crop data...
  > Interpolating missing irrigation records...
  > Calculating Irrigated Fraction (Area / Total Area)...
  > Data Prep Complete.

--- Processing Shift: -1 ---
--- Running Climate Aggregation (Smart Season: [5, 6, 7, 8, 9, 10, 11]) ---


/tmp/ipython-input-702866272.py:18: FutureWarning: updating coordinate 'time', which is a PandasMultiIndex, would leave the multi-index level coordinates ['year', 'month'] in an inconsistent state. This will raise an error in the future. Use `.drop_vars(['time', 'year', 'month'])` to drop the coordinates' values before assigning new coordinate values.
  return da.transpose(y, m, ...).stack(time=(y, m)).assign_coords(time=("time", time)).transpose("time", ...)


--- Generating CROP-WEIGHTED SIF panel (Smart Season) for 'maize' ---


/tmp/ipython-input-702866272.py:177: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  district_means = joined_gdf.groupby('Dist Code').apply(weighted_avg).reset_index(name='csif_season_mean')
/tmp/ipython-input-702866272.py:177: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  district_means = joined_gdf.groupby('Dist Code').apply(weighted_avg).reset_index(name='csif_season_mean')
/tmp/ipython-input-702866272.py:

--- Running Climate Aggregation (Smart Season: [6, 7, 8, 9, 10, 11]) ---


...

--- Running Climate Aggregation (Smart Season: [11, 12, 1, 2, 3]) ---
    [!] Detected Year-Crossing Season (Start: 11). adjusting logic...


/tmp/ipython-input-702866272.py:18: FutureWarning: updating coordinate 'time', which is a PandasMultiIndex, would leave the multi-index level coordinates ['year', 'month'] in an inconsistent state. This will raise an error in the future. Use `.drop_vars(['time', 'year', 'month'])` to drop the coordinates' values before assigning new coordinate values.
  return da.transpose(y, m, ...).stack(time=(y, m)).assign_coords(time=("time", time)).transpose("time", ...)


--- Generating CROP-WEIGHTED SIF panel (Smart Season) for 'sorghum_rabi' ---


/tmp/ipython-input-702866272.py:149: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  combined = xr.concat([data1, data2], dim='month')
/tmp/ipython-input-702866272.py:177: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  district_means = joined_gdf.groupby('Dist Code').apply(weighted_avg).reset_index(name='csif_season_mean')
/tmp/ipython-input-702866272.py:149: Fut


 [✓] SUCCESS: Hierarchical results saved to:
     /content/drive/MyDrive/IndiaProject_Colab/results/FINAL_HIERARCHICAL_INTERACTION_metrics.parquet
     Total Records: 2324

Columns Generated:
['Dist Code', 'state', 'district', 'n_years', 'R2_1_Trend', 'R2_2_Climate', 'R2_3_Irrig', 'R2_4_Full', 'Mq_Climate', 'Mq_Irrig', 'Mq_SIF', 'R2_cv', 'year_shift', 'crop']

Sanity Check (Mean Marginal Gains):
Mq_Climate    0.254726
Mq_Irrig      0.050712
Mq_SIF        0.047953
dtype: float64
